In [79]:
# import snowflake
# from snowflake.snowpark.context import get_active_session
# session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# ----- 1) Imports & Reproducibility -----
import os
import math
import random
import numpy as np
import pandas as pd
import re

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Displaying Data
import matplotlib.pyplot as plt

In [166]:
Water_Quality_df = pd.read_csv("data/water_quality_training_dataset.csv")
landsat_train_features = pd.read_csv("data/landsat_features_training_combined.csv")
Terraclimate_df = pd.read_csv("data/terraclimate_features_training_combined.csv")
depWaterAndSan_df = pd.read_csv("data/depWaterAndSan/depWaterAndSan_features_training_chemReadings_nonnull.csv")

#Clean Up the Data
#Capitalize Col Names
def capitalize_words_keep_spacing(col):
    # Split on space or underscore but keep the separators
    parts = re.split(r'([ _])', col)
    # Capitalize text parts, keep separators unchanged
    return ''.join(p.title() if p not in [' ', '_'] else p for p in parts)
Terraclimate_df.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df.columns]
landsat_train_features.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features.columns]
Water_Quality_df.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df.columns]
depWaterAndSan_df.columns = [capitalize_words_keep_spacing(c) for c in depWaterAndSan_df.columns]

# Fix Dataset
eps = 1e-10
landsat_train_features['Ndmi'] = (landsat_train_features['Nir08'] - landsat_train_features['Swir16']) / (landsat_train_features['Nir08'] + landsat_train_features['Swir16'] + eps)
landsat_train_features['Mndwi'] = (landsat_train_features['Green'] - landsat_train_features['Swir16']) / (landsat_train_features['Green'] + landsat_train_features['Swir16'] + eps)


def combine_two_datasets(dataset1,dataset2,dataset3,dataset4):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2,dataset3,dataset4], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df, depWaterAndSan_df)

#Data type corrections 
wq_data['Sample Date'] = pd.to_datetime(wq_data['Sample Date'],  format='%d-%m-%Y')


def convert_text_cols_to_float(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if (pd.api.types.is_object_dtype(out[c]) 
            or pd.api.types.is_string_dtype(out[c]) 
            or pd.api.types.is_categorical_dtype(out[c])):
            s = out[c].astype(str).str.replace('\u00A0', ' ', regex=False).str.strip()
            s = s.str.replace('\u2212', '-', regex=False)                 # unicode minus
            s = s.str.replace(r'^\(\s*(.*)\s*\)$', r'-\1', regex=True)    # (123) -> -123
            s = s.str.replace(r'^\s*([+]?\s*[\d, .]+)\s*-$', r'-\1', regex=True)  # 123- -> -123
            s = s.str.replace(r'(?<=\d),(?=\d{3}\b)', '', regex=True)      # thousands comma
            out[c] = pd.to_numeric(s, errors='coerce')                     # float64 by default with NaNs
    return out

wq_data = convert_text_cols_to_float(wq_data)

#ullify all negative observations
for column in wq_data.columns:
    if column != "Sample Date": wq_data[wq_data[column] < -9000][column] = np.nan 

wq_data['Month_cosine'] = np.cos((wq_data['Sample Date'].dt.month + (wq_data['Sample Date'].dt.day/31))* np.pi / 6)

#wq_data = wq_data.drop(columns=['Qa_Radsat', 'Cloud_Qa', 'Sample Date',  'Atmos_Opacity', 'Emsd', 'Lwir', 'Unnamed: 0'])
#, 'Atmos opacity, 'Emsd', 'Lwir''
feature_cols = wq_data.columns.tolist()

#print(wq_data.columns)
feature_cols.remove('Latitude')
feature_cols.remove('Longitude')
feature_cols.remove('Month_cosine')
feature_cols.remove('Total Alkalinity')
feature_cols.remove('Electrical Conductance')
feature_cols.remove('Dissolved Reactive Phosphorus')

wq_data = wq_data.dropna(subset=feature_cols)

wq_data = wq_data.drop(columns='Sample Date')

In [ ]:
wq_data = wq_data.drop(columns=[
    # # Redundant spectral/ratios
    # "Osavi", "Savi", "Gcvi", "Gndvi",
    # "Green/Nir Ratio", "Red/Nir Ratio", "Green/Red Ratio", "Ndgi",
    # # Dryness / urban axis
    # "Ndmi", "Ndbi", "Ui (Urban Index)", "Bsi",
    # # Bands / thermal
    # "Swir16", "Urad",
    # # Hydro-met redundancy / leakage
    # "Ppt", "Aet", "Vpd",
    # we just need one of dl and non dl set
    "Na_Diss_Water_Dl", 
    "Cl_Diss_Water_Dl", 
    "Ca_Diss_Water_Dl", 
    "Mg_Diss_Water_Dl", 
    "So4_Diss_Water_Dl", 
    "F_Diss_Water_Dl", 
    "Tal_Diss_Water_Dl", 
    "Ph_Diss_Water_Dl", 
    "No3_No2_N_Diss_Water_Dl", 
    "Nh4_N_Diss_Water_Dl", 
    "Po4_P_Diss_Water_Dl", 
    "Ec_Phys_Water_Dl"
])

In [169]:
wq_data_mini = wq_data[['Latitude', 'Longitude', 'Month_cosine', 'Total Alkalinity',
      'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'Swir22', 'Ndmi', 'Mndwi', 'Pet', 'Aet', 'Soil', 'Def']]

In [193]:
Water_Quality_df_v = pd.read_csv("data/submission_template.csv")
landsat_train_features_v = pd.read_csv("data/landsat_features_validation_combined.csv")
Terraclimate_df_v = pd.read_csv("data/terraclimate_features_validation_combined.csv")
depWaterAndSan_df_v = pd.read_csv("data/depWaterAndSan/depWaterAndSan_features_validation_chemReadings_nonnull.csv")

#Clean Up the Data
Terraclimate_df_v.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df_v.columns]
landsat_train_features_v.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features_v.columns]
Water_Quality_df_v.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df_v.columns]
depWaterAndSan_df_v.columns = [capitalize_words_keep_spacing(c) for c in depWaterAndSan_df_v.columns]

#print(landsat_train_features_v.columns)

# Fix Dataset
eps = 1e-10
landsat_train_features_v['Ndmi'] = (landsat_train_features_v['Nir08'] - landsat_train_features_v['Swir16']) / (landsat_train_features_v['Nir08'] + landsat_train_features_v['Swir16'] + eps)
landsat_train_features_v['Mndwi'] = (landsat_train_features_v['Green'] - landsat_train_features_v['Swir16']) / (landsat_train_features_v['Green'] + landsat_train_features_v['Swir16'] + eps)


wq_data_v = combine_two_datasets(Water_Quality_df_v, landsat_train_features_v, Terraclimate_df_v, depWaterAndSan_df_v)



#Data type corrections 
wq_data_v['Sample Date'] = pd.to_datetime(wq_data_v['Sample Date'],  format='%d-%m-%Y')
wq_data_v = convert_text_cols_to_float(wq_data_v)



#wq_data_v = wq_data_v.drop(columns=['Qa_Radsat', 'Cloud_Qa', 'Atmos_Opacity', 'Emsd', 'Lwir', 'Unnamed: 0'])
#ullify all negative observations
for column in wq_data_v.columns:
    if column != "Sample Date": wq_data_v[wq_data_v[column] < -9000][column] = np.nan 



wq_data_v['Month_cosine'] = np.cos((wq_data_v['Sample Date'].dt.month + (wq_data_v['Sample Date'].dt.day/31))* np.pi / 6)

#print(wq_data_v.shape)

wq_data_final = wq_data_v[['Latitude', 'Longitude', 'Sample Date', 'Month_cosine']]

#display(wq_data_v.isna().sum().to_frame().T)

wq_data_v = wq_data_v.dropna(subset=feature_cols)

#print(wq_data_v.shape)

wq_data_v = wq_data_v.drop(columns='Sample Date')

wq_data_v = wq_data_v[sorted(wq_data_v.columns)]

#print(wq_data_final.columns)

In [ ]:
wq_data_v = wq_data_v.drop(columns=[
    # # Redundant spectral/ratios
    # "Osavi", "Savi", "Gcvi", "Gndvi",
    # "Green/Nir Ratio", "Red/Nir Ratio", "Green/Red Ratio", "Ndgi",
    # # Dryness / urban axis
    # "Ndmi", "Ndbi", "Ui (Urban Index)", "Bsi",
    # # Bands / thermal
    # "Swir16", "Urad",
    # # Hydro-met redundancy / leakage
    # "Ppt", "Aet", "Vpd",
    # we just need one of dl and non dl set
    "Na_Diss_Water_Dl", 
    "Cl_Diss_Water_Dl", 
    "Ca_Diss_Water_Dl", 
    "Mg_Diss_Water_Dl", 
    "So4_Diss_Water_Dl", 
    "F_Diss_Water_Dl", 
    "Tal_Diss_Water_Dl", 
    "Ph_Diss_Water_Dl", 
    "No3_No2_N_Diss_Water_Dl", 
    "Nh4_N_Diss_Water_Dl", 
    "Po4_P_Diss_Water_Dl", 
    "Ec_Phys_Water_Dl"
])

In [194]:
wq_data_v_mini = wq_data_v[['Latitude', 'Longitude', 'Month_cosine', 'Total Alkalinity',
       'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'Swir22', 'Ndmi', 'Mndwi', 'Pet', 'Aet', 'Soil', 'Def']]

In [196]:
#number of cv groups
cv_groups = 8

# #split over longitude
# wq_data_mini['cv_group'] = pd.qcut(wq_data_mini['Latitude'], q=cv_groups, labels=False)

# # Specify the number of folds
# lat_sep_kf = GroupKFold(n_splits=cv_groups - 1)

In [ ]:
def add_spatial_clusters(
    df: pd.DataFrame,
    lat_col: str = "Latitude",
    lon_col: str = "Longitude",
    n_clusters: int = cv_groups,
    group_col: str = "cv_group",
    random_state: int = 88
) -> pd.DataFrame:
    """
    Create spatial groups by clustering latitude/longitude.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    lat_col : str
        Name of latitude column.
    lon_col : str
        Name of longitude column.
    n_clusters : int
        Number of spatial clusters to create.
    group_col : str
        Name of output cluster/group column.
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    pd.DataFrame
        Copy of df with an added spatial grouping column.
    """

    out = df.copy()

    # Ensure numeric coordinates
    out[lat_col] = pd.to_numeric(out[lat_col], errors="coerce")
    out[lon_col] = pd.to_numeric(out[lon_col], errors="coerce")

    # Identify rows with valid coordinates
    valid_mask = out[lat_col].notna() & out[lon_col].notna()

    if valid_mask.sum() == 0:
        raise ValueError("No valid latitude/longitude pairs found for clustering.")

    # Extract valid coordinates
    lat = out.loc[valid_mask, lat_col].to_numpy()
    lon = out.loc[valid_mask, lon_col].to_numpy()

    # Convert degrees to approximate km coordinates
    # Latitude: ~111.32 km per degree
    # Longitude: scaled by cos(latitude)
    lat_km = lat * 111.32
    lon_km = lon * 111.32 * np.cos(np.radians(lat))

    coords_km = np.column_stack([lat_km, lon_km])

    # Fit clustering
    km = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=20
    )
    labels = km.fit_predict(coords_km)

    # Assign labels back to dataframe
    out[group_col] = np.nan
    out.loc[valid_mask, group_col] = labels
    out[group_col] = out[group_col].astype("Int64")

    return out

wq_data_mini = add_spatial_clusters(
    wq_data_mini,
    lat_col="Latitude",
    lon_col="Longitude",
    n_clusters=cv_groups,
    group_col="cv_group"
)

In [197]:
#test train based on location
wq_data_test = wq_data_mini[wq_data_mini['cv_group'] == 0]
wq_data_train = wq_data_mini[wq_data_mini['cv_group'] != 0]

wq_data_test = wq_data_test.drop(columns='cv_group')
wq_data_train = wq_data_train.drop(columns='cv_group')

wq_data_train = wq_data_train[sorted(wq_data_train.columns)]
wq_data_test = wq_data_test[sorted(wq_data_test.columns)]

#then split into X and Y 
#Y_train = wq_data_train[["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]]
#X_train = wq_data_train.drop(columns=["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus", "Longitude", "Latitude"])
#Y_test = wq_data_test[["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]]
#X_test = wq_data_test.drop(columns=["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"])

In [199]:
# For reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


In [200]:
# ----- 2) Configurable column names & task type -----
# Adjust these lists for your real dataset
key_cols     = ["Latitude", "Longitude", "Month_cosine"]  # 3 key columns (IDs, timestamps, etc.)
feature_cols = wq_data_mini.columns.tolist()
feature_cols.remove('Latitude')
feature_cols.remove('Longitude')
feature_cols.remove('Month_cosine')
feature_cols.remove('Total Alkalinity')
feature_cols.remove('Electrical Conductance')
feature_cols.remove('Dissolved Reactive Phosphorus')
feature_cols.remove('cv_group')
#feature_cols = [f"f{i}" for i in range(1, 21)]  # 20 feature columns
target_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
#target_cols  = ["y1", "y2", "y3"]  # 3 target columns

# Choose the task: "regression" (default) or "multilabel"
#  - regression:  continuous y1,y2,y3  -> MSELoss
#  - multilabel:  binary y1,y2,y3 in {0,1} -> BCEWithLogitsLoss
task_type = "regression"

In [ ]:
wq_data_train.shape

In [ ]:
wq_data_v_mini.shape

In [ ]:
for col in wq_data_v_mini.columns:
    if col not in wq_data_train.columns:
        print('Missing ', col)

for col in wq_data_train.columns:
    if col not in wq_data_v_mini.columns:
        print('Missing ', col)
        


In [ ]:
# 1) Fit on TRAIN FEATURES ONLY
scaler = StandardScaler()
scaler.fit(wq_data_train[feature_cols])

# 2) Helper to return a DataFrame with scaled features but keys/targets intact
def apply_scaler(df, key_cols, feature_cols, target_cols, scaler):
    df_out = df.copy()
    X_scaled = scaler.transform(df_out[feature_cols])
    df_out.loc[:, feature_cols] = X_scaled  # replace only features
    return df_out

df_train_n = apply_scaler(wq_data_train, key_cols, feature_cols, target_cols, scaler)
df_val_n   = apply_scaler(wq_data_test,  key_cols, feature_cols, target_cols, scaler)
df_test_n  = apply_scaler(wq_data_v_mini, key_cols, feature_cols, target_cols, scaler)

# Now this works because df_*_n are DataFrames
train_ds = TabularDataset(df_train_n, key_cols, feature_cols, target_cols, task_type=task_type)
val_ds   = TabularDataset(df_val_n,   key_cols, feature_cols, target_cols, task_type=task_type)
test_ds  = TabularDataset(df_test_n,  key_cols, feature_cols, target_cols, task_type=task_type)

In [204]:
# ----- 8) DataLoaders -----
BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [205]:
# ----- 9) Model: a simple MLP -----
class MLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, hidden_dims=(128, 64, 32), dropout=0.1, task_type="regression"):
        super().__init__()
        layers = []
        last = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        layers.append(nn.Linear(last, out_dim))
        self.net = nn.Sequential(*layers)
        self.task_type = task_type

    def forward(self, x):
        logits_or_outputs = self.net(x)
        # For regression, return raw outputs.
        # For multilabel classification: during training we keep raw logits (BCEWithLogitsLoss expects logits).
        return logits_or_outputs

model = MLP(in_dim=len(feature_cols), out_dim=len(target_cols), hidden_dims=(64, 32), dropout=0.15, task_type=task_type).to(DEVICE)


In [206]:
# ----- 10) Loss and Optimizer -----
if task_type == "regression":
    criterion = nn.MSELoss()
else:  # "multilabel"
    criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

In [207]:
# ----- 11) Metrics -----
@torch.no_grad()
def evaluate(model, loader, task_type="regression", threshold=0.5):

    preds_list = []
    targets_list = []


    model.eval()
    total_loss = 0.0
    n_examples = 0

    # For regression: track MAE and RMSE
    mae_sum = torch.zeros(len(target_cols), device=DEVICE)
    mse_sum = torch.zeros(len(target_cols), device=DEVICE)

    # For multilabel: track accuracy (micro)
    correct = 0
    total   = 0

    for _, X, y in loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        out = model(X)

        preds_list.append(out.detach().cpu())
        targets_list.append(y.detach().cpu())

        loss = criterion(out, y)

        if torch.isnan(loss) or torch.isinf(loss):
            print("Bad loss detected!")
            print("out min/max:", out.min().item(), out.max().item())
            print("y min/max:", y.min().item(), y.max().item())
            print("loss:", loss)
            return {"loss": float("nan")}

        bs = y.size(0)
        total_loss += loss.item() * bs
        n_examples += bs

        if task_type == "regression":
            err = torch.abs(out - y)
            mae_sum += err.sum(dim=0)
            mse_sum += ((out - y) ** 2).sum(dim=0)
        else:
            # multilabel: apply sigmoid then threshold
            probs = torch.sigmoid(out)
            preds = (probs >= threshold).float()
            correct += (preds == y).sum().item()
            total   += y.numel()

    avg_loss = total_loss / max(n_examples, 1)

    if task_type == "regression":

        preds = torch.cat(preds_list, dim=0)
        targets = torch.cat(targets_list, dim=0)

        mae = (mae_sum / n_examples).detach().cpu().numpy()
        rmse = torch.sqrt(mse_sum / n_examples).detach().cpu().numpy()

        # Compute R2 per target
        ss_res = torch.sum((preds - targets) ** 2, dim=0)
        ss_tot = torch.sum((targets - targets.mean(dim=0)) ** 2, dim=0)

        r2 = 1 - ss_res / torch.clamp(ss_tot, min=1e-12)
        r2 = r2.numpy()

        metrics = {
            "loss": avg_loss,
            "MAE_per_target": {t: float(mae[i]) for i, t in enumerate(target_cols)},
            "RMSE_per_target": {t: float(rmse[i]) for i, t in enumerate(target_cols)},
            "R2_per_target": {t: float(r2[i]) for i, t in enumerate(target_cols)},
        }
    else:
        accuracy = correct / max(total, 1)
        metrics = {"loss": avg_loss, "micro_accuracy": accuracy}

    return metrics

In [208]:
# ----- 12) Training Loop -----
EPOCHS = 15
best_val = math.inf
patience = 10
wait = 0
best_state = None

epo_tab = []
train_loss_tab = []
train_val_loss_tab = []


for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    n_seen = 0

    for _, X, y in train_loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        out = model(X)
        
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        n_seen += X.size(0)

    train_loss = running_loss / max(n_seen, 1)
    val_metrics = evaluate(model, val_loader, task_type=task_type)
    val_loss = val_metrics["loss"]

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Metrics: {val_metrics}")
    epo_tab.append(epoch)
    train_loss_tab.append(train_loss)
    train_val_loss_tab.append(val_loss)

    # Simple early stopping on validation loss
    if val_loss < best_val - 1e-6:
        best_val = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping.")
            break

# Load best model (if available)
if best_state is not None:
    model.load_state_dict(best_state)

In [209]:
# ----- 13) Final evaluation on TEST -----
test_metrics = evaluate(model, val_loader, task_type=task_type)
print("TEST:", test_metrics)

In [ ]:
plt.plot(epo_tab, train_loss_tab, 'r-o', label='Training')
plt.plot(epo_tab, train_val_loss_tab, 'g--s', label='Validation')

plt.title('Comparison of Two Variables')
plt.xlabel('X Variable')
plt.ylabel('Y Variable')
plt.legend()
plt.grid(True)
plt.show()

In [131]:
# ----- 14) Inference: get predictions + keys side-by-side -----
@torch.no_grad()
def predict_with_keys(model, loader, task_type="regression", threshold=0.5):
    model.eval()
    all_rows = []
    for keys, X, _ in loader:
        X = X.to(DEVICE)
        logits_or_outputs = model(X).cpu().numpy()

        if task_type == "regression":
            preds = logits_or_outputs  # raw outputs
        else:
            probs = 1 / (1 + np.exp(-logits_or_outputs))  # sigmoid
            preds = (probs >= threshold).astype(np.float32)

        # For each batch row, combine keys + predictions into a dict
        for i in range(len(keys["Latitude"])):
            row = {k: keys[k][i] for k in keys.keys()}
            for j, t in enumerate(target_cols):
                row[f"pred_{t}"] = float(preds[i, j])
            all_rows.append(row)
    return pd.DataFrame(all_rows)

pred_df = predict_with_keys(model, test_loader, task_type=task_type)
pred_df.head(5)

In [132]:
pred_df['Latitude'] = pred_df['Latitude'].apply(lambda x: x.item())
pred_df['Longitude'] = pred_df['Longitude'].apply(lambda x: x.item())
pred_df['Month_cosine'] = pred_df['Month_cosine'].apply(lambda x: x.item())

In [133]:
# Reverse cosine encoding
def reverse_month_cosine(cos_val):
    # Ensure value is in valid range for arccos
    cos_val = np.clip(cos_val, -1, 1)
    
    # Get angle in radians
    angle = np.arccos(cos_val)
    
    # Convert back to fractional month
    fractional_month = angle / (np.pi / 6)  # inverse of * np.pi/6
    
    # Approximate month and day
    month = int(fractional_month)  # integer month
    day = int(round((fractional_month - month) * 31))  # approximate day
    
    # Adjust for month starting at 1
    month = max(1, min(12, month))
    day = max(1, min(31, day))
    
    return month, day

# # Apply to DataFrame
# pred_df[['Approx_Month', 'Approx_Day']] = pred_df['Month_cosine'].apply(
#     lambda x: pd.Series(reverse_month_cosine(x))
# )

In [134]:
display(pred_df)

In [136]:
condition = (pred_df['pred_Total Alkalinity'] > 20) & (pred_df['pred_Total Alkalinity'] < 200)

count1 = condition.sum()

print("Count using sum():", count1)


condition = (pred_df['pred_Electrical Conductance'] < 800)

count1 = condition.sum()

print("Count using sum():", count1)

condition = (pred_df['pred_Dissolved Reactive Phosphorus'] < 100)

count1 = condition.sum()

print("Count using sum():", count1)

In [ ]:
final_df = wq_data_final.merge(pred_df, on=['Latitude', 'Longitude', 'Month_cosine'], how='inner')

In [ ]:
final_df.info()

In [135]:
pred_df.to_csv('data/output.csv', index=False)

In [ ]:
session.sql("""
    PUT file://data/output.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge-version2"/versions/live/data/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")